# Evaluation Results of RAG

First, import all required libraries.

In [ ]:
from src.loaders import parse_pdf_folder
from src.chunking import split_text
from src.embeddings import build_embedding_model
from src.vectorstore import build_database, save_faiss_index, load_faiss_index
from src.retriever import build_similarity_retriever
from src.evaluation import load_eval_data, evaluate_retrieval, summarize_retrieval_results, save_evaluation_results
from src.reranking import build_rerank_retriever
from src.config import RERANK_BASE_K, RAW_PDF_DIR

Load embedding model:

In [ ]:
embedding_model = build_embedding_model("miniLM")

### 1.1.A Build Database

In [ ]:
documents = parse_pdf_folder(RAW_PDF_DIR)
chunked_documents = split_text(documents)
vectorstore = build_database(chunked_documents, embedding_model)
save_faiss_index(vectorstore, "faiss_minilm")

### 1.1.B Load Database

In [ ]:
vectorstore = load_faiss_index("faiss_minilm", embedding_model)

## 1. Retriever Only
First, let's evaluate the RAG with retriever only.

### 1.2. Build Retriever

In [ ]:
base_retriever = build_similarity_retriever(vectorstore, k=RERANK_BASE_K)

In [ ]:
eval_data = load_eval_data("Evaluation_QA_2.json")

In [ ]:
base_results = evaluate_retrieval(eval_data, base_retriever)
base_summary = summarize_retrieval_results(base_results)

print(base_summary)

save_evaluation_results(
    base_results,
    base_summary,
    "baseline_retrieval_results.json"
)

## 2. With Reranker

Now, let's add reranker to the pipeline and test the results.

In [ ]:
rerank_retriever = build_rerank_retriever(base_retriever)

In [ ]:
rerank_results = evaluate_retrieval(eval_data, rerank_retriever)
rerank_summary = summarize_retrieval_results(rerank_results)

print("Rerank summary:", rerank_summary)

save_evaluation_results(
    rerank_results,
    rerank_summary,
    "rerank_retrieval_results.json"
)